In [1]:
!pip install matplotlib pandas seaborn numpy nltk

In [2]:
%matplotlib inline

import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import _pickle as cPickle
import nltk

## Prediction (baseline)

In [5]:
df_start = pd.read_csv('df_start.tsv', sep=',')

In [6]:
df_start

,interaction_id,user_id,session_id,timestamp,app_name,event_type,time_dff,open_time,close_time
0,1,0,1,2018-01-16 06:01:05,Minesweeper Classic (Mines),Opened,NaN,2018-01-16 06:01:05,2018-01-16 06:01:09
1,2,0,1,2018-01-16 06:03:44,Minesweeper Classic (Mines),Opened,0 days 00:02:35,2018-01-16 06:03:44,2018-01-16 06:04:17
2,3,0,2,2018-01-16 06:25:54,Gmail,User Interaction,0 days 00:21:37,2018-01-16 06:25:54,2018-01-16 06:25:54
3,4,0,2,2018-01-16 06:26:05,Google,Opened,0 days 00:00:11,2018-01-16 06:26:05,2018-01-16 06:26:10
4,5,0,2,2018-01-16 06:26:10,Instagram,Opened,0 days 00:00:00,2018-01-16 06:26:10,2018-01-16 06:26:21
...,...,...,...,...,...,...,...,...,...
599630,599631,291,76245,2018-04-06 12:47:28,Facebook Messenger,Opened,0 days 00:03:29,2018-04-06 12:47:28,2018-04-06 12:53:13
599631,599632,291,76246,2018-04-06 13:20:12,Settings,Opened,0 days 00:26:59,2018-04-06 13:20:12,2018-04-06 13:20:14
599632,599633,291,76247,2018-04-06 14:34:15,Settings,Opened,0 days 01:14:01,2018-04-06 14:34:15,2018-04-06 14:34:17
599633,599634,291,76247,2018-04-06 14:34:34,Facebook,Opened,0 days 00:00:17,2018-04-06 14:34:34,2018-04-06 14:35:37


### Setup (grouping users)

In [7]:
# for each user create its own dataframe
df_user = df_start.groupby('user_id')

In [8]:
# print 10 rows of first user
df_user.get_group(0).head(10)

,interaction_id,user_id,session_id,timestamp,app_name,event_type,time_dff,open_time,close_time
0,1,0,1,2018-01-16 06:01:05,Minesweeper Classic (Mines),Opened,NaN,2018-01-16 06:01:05,2018-01-16 06:01:09
1,2,0,1,2018-01-16 06:03:44,Minesweeper Classic (Mines),Opened,0 days 00:02:35,2018-01-16 06:03:44,2018-01-16 06:04:17
2,3,0,2,2018-01-16 06:25:54,Gmail,User Interaction,0 days 00:21:37,2018-01-16 06:25:54,2018-01-16 06:25:54
3,4,0,2,2018-01-16 06:26:05,Google,Opened,0 days 00:00:11,2018-01-16 06:26:05,2018-01-16 06:26:10
4,5,0,2,2018-01-16 06:26:10,Instagram,Opened,0 days 00:00:00,2018-01-16 06:26:10,2018-01-16 06:26:21
5,6,0,2,2018-01-16 06:26:21,Google,Opened,0 days 00:00:00,2018-01-16 06:26:21,2018-01-16 06:26:26
6,7,0,2,2018-01-16 06:26:26,Google Chrome,Opened,0 days 00:00:00,2018-01-16 06:26:26,2018-01-16 06:36:26
7,8,0,4,2018-01-16 07:15:56,Minesweeper Classic (Mines),Opened,0 days 00:39:30,2018-01-16 07:15:56,2018-01-16 07:15:58
8,9,0,4,2018-01-16 07:20:45,Google Chrome,Opened,0 days 00:04:47,2018-01-16 07:20:45,2018-01-16 07:20:45
9,10,0,4,2018-01-16 07:20:46,Minesweeper Classic (Mines),Opened,0 days 00:00:01,2018-01-16 07:20:46,2018-01-16 07:21:43


In [9]:
# for user 0 print all unique app names
df_user.get_group(0)['app_name'].unique()

array(['Minesweeper Classic (Mines)', 'Gmail', 'Google', 'Instagram',
       'Google Chrome', 'Clock', 'Maps', 'YouTube', 'Facebook',
       'Messages', 'Phone', 'Snapchat', 'Settings', 'Google Photos',
       'Hangouts', 'Amazon Shopping', 'Facebook Messenger',
       'Google Play Store', 'Calendar'], dtype=object)

### Random app (predict next app as random app that user uses)

In [11]:
# create user to app dictionary including all unique app names for each user
user_to_app_dict = {}

for user_id, user_df in df_user:
    print(f"Saving unique apps for user {user_id}")
    user_to_app_dict[user_id] = user_df['app_name'].unique()

# print user to app dictionary
print("User to app dictionary: ")
user_to_app_dict


Saving unique apps for user 0
Saving unique apps for user 1
Saving unique apps for user 2
Saving unique apps for user 3
Saving unique apps for user 4
Saving unique apps for user 5
Saving unique apps for user 6
Saving unique apps for user 7
Saving unique apps for user 8
Saving unique apps for user 9
Saving unique apps for user 10
Saving unique apps for user 11
Saving unique apps for user 12
Saving unique apps for user 13
Saving unique apps for user 14
Saving unique apps for user 15
Saving unique apps for user 16
Saving unique apps for user 17
Saving unique apps for user 18
Saving unique apps for user 19
Saving unique apps for user 20
Saving unique apps for user 21
Saving unique apps for user 22
Saving unique apps for user 23
Saving unique apps for user 24
Saving unique apps for user 25
Saving unique apps for user 26
Saving unique apps for user 27
Saving unique apps for user 28
Saving unique apps for user 29
Saving unique apps for user 30
Saving unique apps for user 31
Saving unique apps

{0: array(['Minesweeper Classic (Mines)', 'Gmail', 'Google', 'Instagram',
        'Google Chrome', 'Clock', 'Maps', 'YouTube', 'Facebook',
        'Messages', 'Phone', 'Snapchat', 'Settings', 'Google Photos',
        'Hangouts', 'Amazon Shopping', 'Facebook Messenger',
        'Google Play Store', 'Calendar'], dtype=object),
 1: array(['Google Photos', 'Discord', 'Messages', 'Google', 'Snapchat',
        'Instagram', 'Google Chrome', 'Facebook Messenger', 'Facebook',
        'Google Drive', 'Twitter', 'Google Play Store', 'YouTube', 'Phone',
        'Spotify Music', 'Clock', 'Settings', 'Reddit'], dtype=object),
 2: array(['Android In Call UI', 'Receipt Hog', 'Gmail', 'Facebook',
        'Google Chrome', 'Instagram', 'Amazon Shopping', 'Google',
        'Snapchat', 'Maps', 'Ibotta', 'YouTube', 'Spotify Music',
        'Google Play Store', 'Settings', 'Facebook Messenger'],
       dtype=object),
 3: array(['PayPal Mobile Cash', 'Google Play Store', 'Google Chrome',
        'YouTube', 'A

In [12]:
# iterate through all df_start interactions and choose app_name as random app from user_to_app_dict for user_id
df_start['random_app_name'] = df_start.apply(lambda row: np.random.choice(user_to_app_dict[row['user_id']]), axis=1)
df_start.head(100)

,interaction_id,user_id,session_id,timestamp,app_name,event_type,time_dff,open_time,close_time,random_app_name
0,1,0,1,2018-01-16 06:01:05,Minesweeper Classic (Mines),Opened,NaN,2018-01-16 06:01:05,2018-01-16 06:01:09,Snapchat
1,2,0,1,2018-01-16 06:03:44,Minesweeper Classic (Mines),Opened,0 days 00:02:35,2018-01-16 06:03:44,2018-01-16 06:04:17,Phone
2,3,0,2,2018-01-16 06:25:54,Gmail,User Interaction,0 days 00:21:37,2018-01-16 06:25:54,2018-01-16 06:25:54,Phone
3,4,0,2,2018-01-16 06:26:05,Google,Opened,0 days 00:00:11,2018-01-16 06:26:05,2018-01-16 06:26:10,Google Chrome
4,5,0,2,2018-01-16 06:26:10,Instagram,Opened,0 days 00:00:00,2018-01-16 06:26:10,2018-01-16 06:26:21,Amazon Shopping
...,...,...,...,...,...,...,...,...,...,...
95,96,0,31,2018-01-16 22:32:56,Gmail,Closed,0 days 00:00:01,2018-01-16 22:32:56,2018-01-16 22:32:56,Settings
96,97,0,31,2018-01-16 22:32:57,Phone,Opened,0 days 00:00:01,2018-01-16 22:32:57,2018-01-16 22:34:07,Google
97,98,0,31,2018-01-16 22:34:07,Gmail,Opened,0 days 00:00:00,2018-01-16 22:34:07,2018-01-16 22:34:47,Snapchat
98,99,0,31,2018-01-16 22:34:47,Google,Opened,0 days 00:00:00,2018-01-16 22:34:47,2018-01-16 22:35:22,Google Chrome


### Evaluation

In [14]:
# create dicitonary user to accuracy
user_to_accuracy_dict = {}

# go through all interactions and check if app_name is in random_app_name for each user and save accuracy
for user_id, user_df in df_user:
    # iterate through all interactions for user_id
    accuracy = 0

    for index, row in user_df.iterrows():
        if row['app_name'] == row['random_app_name']:
            accuracy += 1

    user_to_accuracy_dict[user_id] = accuracy / len(user_df)

# print user to accuracy dictionary
print("Random prediction: user to accuracy prediction")
user_to_accuracy_dict

Random prediction: user to accuracy prediction


{0: 0.048162230671736375,
 1: 0.042986425339366516,
 2: 0.07391304347826087,
 3: 0.08284023668639054,
 4: 0.13157894736842105,
 5: 0.029397985476692434,
 6: 0.045376220562894885,
 7: 0.045627376425855515,
 8: 0.03333333333333333,
 9: 0.053881278538812784,
 10: 0.1118421052631579,
 11: 0.060371959942775395,
 12: 0.05138339920948617,
 13: 0.04456824512534819,
 14: 0.1276595744680851,
 15: 0.046511627906976744,
 16: 0.04468599033816425,
 17: 0.05555555555555555,
 18: 0.04115410422724223,
 19: 0.04597701149425287,
 20: 0.08536585365853659,
 21: 0.049315068493150684,
 22: 0.036684782608695655,
 23: 0.07647058823529412,
 24: 0.13131313131313133,
 25: 0.044018643190056966,
 26: 0.053769900871132474,
 27: 0.06716417910447761,
 28: 0.04608294930875576,
 29: 0.09727626459143969,
 30: 0.10738255033557047,
 31: 0.09242982704848313,
 32: 0.15517241379310345,
 33: 0.062328498627989025,
 34: 0.06818181818181818,
 35: 0.10714285714285714,
 36: 0.06425153793574846,
 37: 0.04873949579831933,
 38: 0.0736

#### Results

In [15]:
# Calculate mean and standard deviation of accuracy
mean_accuracy = np.mean(list(user_to_accuracy_dict.values()))
std_accuracy = np.std(list(user_to_accuracy_dict.values()))

print("Mean accuracy & standard deviation: ")
mean_accuracy, std_accuracy

Mean accuracy & standard deviation: 


(0.09737911663182187, 0.1461736360730588)

### Most popular app (predict next app as user's favorite app)

In [16]:
# create user most popular app dictionary
user_to_most_popular_app_dict = {}

# for each user iterate through all interactions and count app_name occurences
for user_id, user_df in df_user:
    # iterate through all interactions for user_id
    app_name_to_count_dict = {}

    for index, row in user_df.iterrows():
        if row['app_name'] in app_name_to_count_dict:
            app_name_to_count_dict[row['app_name']] += 1
        else:
            app_name_to_count_dict[row['app_name']] = 1

    print('user_id: ', user_id)
    print('app_name_to_count_dict: ', app_name_to_count_dict)
    # find most popular app_name for user_id
    most_popular_app_name = max(app_name_to_count_dict, key=app_name_to_count_dict.get)
    user_to_most_popular_app_dict[user_id] = most_popular_app_name

# print user to most popular app dictionary
print("User to most popular app dictionary: ")
user_to_most_popular_app_dict

user_id:  0
app_name_to_count_dict:  {'Minesweeper Classic (Mines)': 174, 'Gmail': 50, 'Google': 170, 'Instagram': 21, 'Google Chrome': 117, 'Clock': 57, 'Maps': 39, 'YouTube': 3, 'Facebook': 20, 'Messages': 70, 'Phone': 35, 'Snapchat': 2, 'Settings': 6, 'Google Photos': 4, 'Hangouts': 1, 'Amazon Shopping': 3, 'Facebook Messenger': 9, 'Google Play Store': 7, 'Calendar': 1}
user_id:  1
app_name_to_count_dict:  {'Google Photos': 4, 'Discord': 80, 'Messages': 15, 'Google': 84, 'Snapchat': 55, 'Instagram': 19, 'Google Chrome': 46, 'Facebook Messenger': 34, 'Facebook': 18, 'Google Drive': 1, 'Twitter': 13, 'Google Play Store': 8, 'YouTube': 6, 'Phone': 9, 'Spotify Music': 35, 'Clock': 2, 'Settings': 2, 'Reddit': 11}
user_id:  2
app_name_to_count_dict:  {'Android In Call UI': 12, 'Receipt Hog': 6, 'Gmail': 62, 'Facebook': 16, 'Google Chrome': 73, 'Instagram': 15, 'Amazon Shopping': 6, 'Google': 6, 'Snapchat': 9, 'Maps': 4, 'Ibotta': 2, 'YouTube': 1, 'Spotify Music': 2, 'Google Play Store': 5

{0: 'Minesweeper Classic (Mines)',
 1: 'Google',
 2: 'Google Chrome',
 3: 'Google Chrome',
 4: 'Google Chrome',
 5: 'Twitter',
 6: 'Reddit',
 7: 'Google',
 8: 'Words With Friends 2',
 9: 'Google Chrome',
 10: 'Google Chrome',
 11: 'Samsung Internet Browser',
 12: 'WhatsApp Messenger',
 13: 'Facebook',
 14: 'Google Chrome',
 15: 'Gmail',
 16: 'Slidejoy',
 17: 'Discord',
 18: 'Google Chrome',
 19: 'Messages',
 20: 'Google Chrome',
 21: 'Samsung Email',
 22: 'Facebook',
 23: 'Samsung Email',
 24: 'Google',
 25: 'Google Chrome',
 26: 'Android In Call UI',
 27: 'Google Chrome',
 28: 'Google Chrome',
 29: 'Google Chrome',
 30: 'Contacts',
 31: 'Telegram',
 32: 'Google',
 33: 'Samsung Internet Browser',
 34: 'Google Chrome',
 35: 'Google Chrome',
 36: 'YouTube',
 37: 'Gmail',
 38: 'Yahoo Mail',
 39: 'Google',
 40: 'Facebook Messenger',
 41: 'Facebook',
 42: 'Google',
 43: 'Google Chrome',
 44: 'Google Chrome',
 45: 'Google Play Store',
 46: 'Contacts',
 47: 'Google Chrome',
 48: 'Twitter',
 4

### Evaluation

In [17]:
# iterate through all df_start interactions and choose app_name as most popular app for user_id
df_start['most_popular_app_name'] = df_start.apply(lambda row: user_to_most_popular_app_dict[row['user_id']], axis=1)

# create dicitonary user to accuracy
user_to_accuracy_dict = {}

# go through all interactions and check if app_name is in random_app_name for each user and save accuracy
for user_id, user_df in df_user:
    # iterate through all interactions for user_id
    accuracy = 0

    for index, row in user_df.iterrows():
        if row['app_name'] == row['most_popular_app_name']:
            accuracy += 1

    user_to_accuracy_dict[user_id] = accuracy / len(user_df)

# print user to accuracy dictionary
print("User to accuracy dictionary: ")
user_to_accuracy_dict

User to accuracy dictionary: 


{0: 0.22053231939163498,
 1: 0.19004524886877827,
 2: 0.3173913043478261,
 3: 0.40236686390532544,
 4: 0.26973684210526316,
 5: 0.3367299133286484,
 6: 0.24717595251771013,
 7: 0.23479087452471484,
 8: 0.36666666666666664,
 9: 0.31689497716894977,
 10: 0.42105263157894735,
 11: 0.3344778254649499,
 12: 0.15019762845849802,
 13: 0.24373259052924792,
 14: 0.48936170212765956,
 15: 0.18733850129198967,
 16: 0.32971014492753625,
 17: 0.29012345679012347,
 18: 0.2516215611719973,
 19: 0.15517241379310345,
 20: 0.4146341463414634,
 21: 0.23835616438356164,
 22: 0.17527173913043478,
 23: 0.19411764705882353,
 24: 0.42424242424242425,
 25: 0.2553081305023304,
 26: 0.2547311504956443,
 27: 0.4253731343283582,
 28: 0.2073732718894009,
 29: 0.39299610894941633,
 30: 0.21476510067114093,
 31: 0.3966543804933371,
 32: 0.5402298850574713,
 33: 0.18659349274794199,
 34: 0.20454545454545456,
 35: 0.2857142857142857,
 36: 0.23513328776486672,
 37: 0.19831932773109243,
 38: 0.1828978622327791,
 39: 0.36

In [18]:
# Calculate mean and standard deviation of accuracy
mean_accuracy = np.mean(list(user_to_accuracy_dict.values()))
std_accuracy = np.std(list(user_to_accuracy_dict.values()))

print("Mean accuracy, standard deviation: ")
mean_accuracy, std_accuracy

Mean accuracy, standard deviation: 


(0.32478872982210727, 0.15336119946355084)

In [22]:
df_start

,interaction_id,user_id,session_id,timestamp,app_name,event_type,time_dff,open_time,close_time,random_app_name,most_popular_app_name
0,1,0,1,2018-01-16 06:01:05,Minesweeper Classic (Mines),Opened,NaN,2018-01-16 06:01:05,2018-01-16 06:01:09,Snapchat,Minesweeper Classic (Mines)
1,2,0,1,2018-01-16 06:03:44,Minesweeper Classic (Mines),Opened,0 days 00:02:35,2018-01-16 06:03:44,2018-01-16 06:04:17,Phone,Minesweeper Classic (Mines)
2,3,0,2,2018-01-16 06:25:54,Gmail,User Interaction,0 days 00:21:37,2018-01-16 06:25:54,2018-01-16 06:25:54,Phone,Minesweeper Classic (Mines)
3,4,0,2,2018-01-16 06:26:05,Google,Opened,0 days 00:00:11,2018-01-16 06:26:05,2018-01-16 06:26:10,Google Chrome,Minesweeper Classic (Mines)
4,5,0,2,2018-01-16 06:26:10,Instagram,Opened,0 days 00:00:00,2018-01-16 06:26:10,2018-01-16 06:26:21,Amazon Shopping,Minesweeper Classic (Mines)
...,...,...,...,...,...,...,...,...,...,...,...
599630,599631,291,76245,2018-04-06 12:47:28,Facebook Messenger,Opened,0 days 00:03:29,2018-04-06 12:47:28,2018-04-06 12:53:13,Gmail,Facebook
599631,599632,291,76246,2018-04-06 13:20:12,Settings,Opened,0 days 00:26:59,2018-04-06 13:20:12,2018-04-06 13:20:14,Facebook Messenger,Facebook
599632,599633,291,76247,2018-04-06 14:34:15,Settings,Opened,0 days 01:14:01,2018-04-06 14:34:15,2018-04-06 14:34:17,Facebook Messenger,Facebook
599633,599634,291,76247,2018-04-06 14:34:34,Facebook,Opened,0 days 00:00:17,2018-04-06 14:34:34,2018-04-06 14:35:37,Facebook,Facebook


### Most recently used (predict next app as the one user most recently used)

In [23]:
df_start['most_recently_used_app'] = 'NOT_PROVIDED'

for i, row in df_start.iterrows():
    if i > 0:
        df_start.at[i, 'most_recently_used_app'] = df_start.at[i - 1, 'app_name']

df_start

,interaction_id,user_id,session_id,timestamp,app_name,event_type,time_dff,open_time,close_time,random_app_name,most_popular_app_name,most_recently_used_app
0,1,0,1,2018-01-16 06:01:05,Minesweeper Classic (Mines),Opened,NaN,2018-01-16 06:01:05,2018-01-16 06:01:09,Snapchat,Minesweeper Classic (Mines),NOT_PROVIDED
1,2,0,1,2018-01-16 06:03:44,Minesweeper Classic (Mines),Opened,0 days 00:02:35,2018-01-16 06:03:44,2018-01-16 06:04:17,Phone,Minesweeper Classic (Mines),Minesweeper Classic (Mines)
2,3,0,2,2018-01-16 06:25:54,Gmail,User Interaction,0 days 00:21:37,2018-01-16 06:25:54,2018-01-16 06:25:54,Phone,Minesweeper Classic (Mines),Minesweeper Classic (Mines)
3,4,0,2,2018-01-16 06:26:05,Google,Opened,0 days 00:00:11,2018-01-16 06:26:05,2018-01-16 06:26:10,Google Chrome,Minesweeper Classic (Mines),Gmail
4,5,0,2,2018-01-16 06:26:10,Instagram,Opened,0 days 00:00:00,2018-01-16 06:26:10,2018-01-16 06:26:21,Amazon Shopping,Minesweeper Classic (Mines),Google
...,...,...,...,...,...,...,...,...,...,...,...,...
599630,599631,291,76245,2018-04-06 12:47:28,Facebook Messenger,Opened,0 days 00:03:29,2018-04-06 12:47:28,2018-04-06 12:53:13,Gmail,Facebook,Facebook Messenger
599631,599632,291,76246,2018-04-06 13:20:12,Settings,Opened,0 days 00:26:59,2018-04-06 13:20:12,2018-04-06 13:20:14,Facebook Messenger,Facebook,Facebook Messenger
599632,599633,291,76247,2018-04-06 14:34:15,Settings,Opened,0 days 01:14:01,2018-04-06 14:34:15,2018-04-06 14:34:17,Facebook Messenger,Facebook,Settings
599633,599634,291,76247,2018-04-06 14:34:34,Facebook,Opened,0 days 00:00:17,2018-04-06 14:34:34,2018-04-06 14:35:37,Facebook,Facebook,Settings


### Evaluation

In [24]:
# create dicitonary user to accuracy
user_to_accuracy_dict = {}

# go through all interactions and check if app_name is in random_app_name for each user and save accuracy
for user_id, user_df in df_user:
    # iterate through all interactions for user_id
    accuracy = 0

    for index, row in user_df.iterrows():
        if row['app_name'] == row['most_recently_used_app']:
            accuracy += 1

    user_to_accuracy_dict[user_id] = accuracy / len(user_df)

# print user to accuracy dictionary
print("User to accuracy dictionary: ")
user_to_accuracy_dict

Most recently used app prediction dictionary: 


{0: 0.28517110266159695,
 1: 0.09728506787330317,
 2: 0.10434782608695652,
 3: 0.27218934911242604,
 4: 0.14473684210526316,
 5: 0.09498711642070742,
 6: 0.08577445912310933,
 7: 0.025665399239543727,
 8: 0.3,
 9: 0.06484018264840183,
 10: 0.3355263157894737,
 11: 0.036337625178826896,
 12: 0.12648221343873517,
 13: 0.08356545961002786,
 14: 0.3829787234042553,
 15: 0.05426356589147287,
 16: 0.14855072463768115,
 17: 0.08641975308641975,
 18: 0.05390292999329009,
 19: 0.05172413793103448,
 20: 0.1951219512195122,
 21: 0.14246575342465753,
 22: 0.06385869565217392,
 23: 0.11176470588235295,
 24: 0.1111111111111111,
 25: 0.022268254790264112,
 26: 0.07209372183838991,
 27: 0.11194029850746269,
 28: 0.04608294930875576,
 29: 0.1575875486381323,
 30: 0.24161073825503357,
 31: 0.17436915225404026,
 32: 0.1206896551724138,
 33: 0.1764014112112897,
 34: 0.23863636363636365,
 35: 0.25,
 36: 0.05673274094326726,
 37: 0.12941176470588237,
 38: 0.0510688836104513,
 39: 0.10790464240903387,
 40: 0

In [25]:
# Calculate mean and standard deviation of accuracy
mean_accuracy = np.mean(list(user_to_accuracy_dict.values()))
std_accuracy = np.std(list(user_to_accuracy_dict.values()))

print("Mean accuracy, standard deviation: ")
mean_accuracy, std_accuracy

Mean accuracy, standard deviation: 


(0.1409909533702817, 0.1396070461366746)